# 05 — Detect Suspicious Logins with IP2Location.io Geolocation

**Use case:** Review account login events for sudden country changes and implausible travel between consecutive successful sign-ins. Enrich unique public IP addresses with the [IP2Location.io REST API](https://www.ip2location.io/ip2location-documentation).

**What you will build:** A repeatable CSV-to-CSV workflow that validates login events, enriches IPs, measures great-circle distance / elapsed time, assigns transparent *review flags*, plots the results, and exports an analyst-ready alert file.

> **Start safely:** `MODE = "demo"` uses deliberately **synthetic** location fixtures with **no network requests**. Locations in demo mode are invented teaching data and do **not** describe the real locations of the sample IP addresses (which belong to public DNS services). Change to `MODE = "live"` for actual API responses.

> **Important:** IP geolocation is approximate. Public DNS servers, VPNs, mobile networks, CDNs, corporate gateways, anycast, and account sharing can produce misleading locations. An impossible-travel flag is a *review signal*, never proof of compromise. Do not automatically block accounts based only on these heuristics.

**Input:** `sample_login_events.csv`  ·  **Outputs:** `outputs_notebook_05/login_events_enriched.csv`, `login_travel_alerts.csv`, and `login_user_summary.csv`.

## 1. Install dependencies

Run once in the active Jupyter kernel. The examples use standard Python, pandas, matplotlib, and requests.

In [ ]:
%pip install -q pandas requests matplotlib

## 2. Configuration

- Keep `MODE = "demo"` for a reproducible example with deliberately constructed geography.
- For real geolocation, change to `MODE = "live"`, substitute actual login IPs in your CSV, and optionally set `IP2LOCATION_IO_API_KEY` in your environment. **Do not paste credentials into notebooks committed to Git.**
- The live example uses individual lookups, not the separate paid bulk endpoint. Each *unique* public IP is requested once, with a conservative cap of `MAX_UNIQUE_IPS`.
- Timestamps must be UTC and include a timezone marker (e.g. `2026-10-20T08:00:00Z`).

In [ ]:
import ipaddress
import math
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import display

MODE = "demo"                     # "demo" or "live"
INPUT_CSV = Path("sample_login_events.csv")
OUTPUT_DIR = Path("outputs_notebook_05")
API_KEY = os.environ.get("IP2LOCATION_IO_API_KEY", "").strip()
API_URL = "https://api.ip2location.io/"
MAX_UNIQUE_IPS = 20               # Guardrail for live API requests
REQUEST_DELAY_SECONDS = 0.15      # Gentle pacing for live requests
REQUEST_TIMEOUT_SECONDS = 15

MIN_DISTANCE_KM = 500             # Only consider substantial jumps
MAX_REASONABLE_SPEED_KMH = 900    # Deliberately adjustable heuristic
MAX_COMPARISON_HOURS = 24         # Ignore stale pairs (not a full travel model)

assert MODE in {"demo", "live"}, "MODE must be 'demo' or 'live'"
print(f"Mode: {MODE} | API key configured: {bool(API_KEY)} | threshold: {MAX_REASONABLE_SPEED_KMH} km/h")

## 3. Prepare input: account login events

The separate `sample_login_events.csv` is included as a download. If it is not present, the notebook writes a **self-contained sample**. All examples are fictional login scenarios; the IP values are public DNS / infrastructure IPs and **not** actual end-user sessions.

CSV schema: `event_id`, `user_id`, `timestamp_utc`, `ip_address`, `outcome` (`success` or `failure`), `user_agent`.

In [ ]:
SAMPLE_EVENTS = [{'event_id': 'evt-001', 'user_id': 'alice', 'timestamp_utc': '2026-10-20T08:00:00Z', 'ip_address': '8.8.8.8', 'outcome': 'success', 'user_agent': 'Chrome / desktop'}, {'event_id': 'evt-002', 'user_id': 'alice', 'timestamp_utc': '2026-10-20T08:20:00Z', 'ip_address': '1.1.1.1', 'outcome': 'success', 'user_agent': 'Chrome / desktop'}, {'event_id': 'evt-003', 'user_id': 'alice', 'timestamp_utc': '2026-10-20T19:00:00Z', 'ip_address': '1.1.1.1', 'outcome': 'success', 'user_agent': 'Chrome / desktop'}, {'event_id': 'evt-004', 'user_id': 'alice', 'timestamp_utc': '2026-10-20T19:03:00Z', 'ip_address': '9.9.9.9', 'outcome': 'failure', 'user_agent': 'Unknown browser'}, {'event_id': 'evt-005', 'user_id': 'alice', 'timestamp_utc': '2026-10-21T15:00:00Z', 'ip_address': '9.9.9.9', 'outcome': 'success', 'user_agent': 'Firefox / laptop'}, {'event_id': 'evt-006', 'user_id': 'bob', 'timestamp_utc': '2026-10-20T07:00:00Z', 'ip_address': '208.67.222.222', 'outcome': 'success', 'user_agent': 'Safari / mobile'}, {'event_id': 'evt-007', 'user_id': 'bob', 'timestamp_utc': '2026-10-20T07:40:00Z', 'ip_address': '4.2.2.2', 'outcome': 'success', 'user_agent': 'Safari / mobile'}, {'event_id': 'evt-008', 'user_id': 'bob', 'timestamp_utc': '2026-10-20T08:10:00Z', 'ip_address': '192.168.1.10', 'outcome': 'success', 'user_agent': 'Safari / mobile'}, {'event_id': 'evt-009', 'user_id': 'bob', 'timestamp_utc': '2026-10-21T12:00:00Z', 'ip_address': '4.2.2.2', 'outcome': 'success', 'user_agent': 'Safari / mobile'}, {'event_id': 'evt-010', 'user_id': 'carol', 'timestamp_utc': '2026-10-20T04:00:00Z', 'ip_address': '9.9.9.9', 'outcome': 'success', 'user_agent': 'Edge / desktop'}, {'event_id': 'evt-011', 'user_id': 'carol', 'timestamp_utc': '2026-10-20T04:55:00Z', 'ip_address': '9.9.9.9', 'outcome': 'success', 'user_agent': 'Edge / desktop'}, {'event_id': 'evt-012', 'user_id': 'carol', 'timestamp_utc': '2026-10-20T05:10:00Z', 'ip_address': '2606:4700:4700::1111', 'outcome': 'success', 'user_agent': 'Edge / desktop'}, {'event_id': 'evt-013', 'user_id': 'carol', 'timestamp_utc': '2026-10-20T05:15:00Z', 'ip_address': 'not-an-ip', 'outcome': 'success', 'user_agent': 'Edge / desktop'}, {'event_id': 'evt-014', 'user_id': 'dave', 'timestamp_utc': '2026-10-20T10:00:00Z', 'ip_address': '1.1.1.1', 'outcome': 'failure', 'user_agent': 'Firefox / desktop'}, {'event_id': 'evt-015', 'user_id': 'dave', 'timestamp_utc': '2026-10-20T10:10:00Z', 'ip_address': '4.2.2.2', 'outcome': 'success', 'user_agent': 'Firefox / desktop'}, {'event_id': 'evt-016', 'user_id': 'dave', 'timestamp_utc': '2026-10-20T12:00:00Z', 'ip_address': '4.2.2.2', 'outcome': 'success', 'user_agent': 'Firefox / desktop'}, {'event_id': 'evt-017', 'user_id': 'erin', 'timestamp_utc': '2026-10-20T06:00:00Z', 'ip_address': '8.8.8.8', 'outcome': 'success', 'user_agent': 'Chrome / desktop'}, {'event_id': 'evt-018', 'user_id': 'erin', 'timestamp_utc': '2026-10-20T06:20:00Z', 'ip_address': '2001:4860:4860::8888', 'outcome': 'success', 'user_agent': 'Chrome / desktop'}, {'event_id': 'evt-019', 'user_id': 'erin', 'timestamp_utc': '2026-10-20T18:00:00Z', 'ip_address': '9.9.9.9', 'outcome': 'success', 'user_agent': 'Chrome / desktop'}, {'event_id': 'evt-020', 'user_id': 'erin', 'timestamp_utc': '2026-10-20T18:20:00Z', 'ip_address': '9.9.9.9', 'outcome': 'success', 'user_agent': 'Chrome / desktop'}]

if not INPUT_CSV.exists():
    pd.DataFrame(SAMPLE_EVENTS).to_csv(INPUT_CSV, index=False)
    print(f"Created sample file: {INPUT_CSV.resolve()}")
else:
    print(f"Reading existing file: {INPUT_CSV.resolve()}")

raw = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)
required = {"event_id", "user_id", "timestamp_utc", "ip_address", "outcome"}
missing = required - set(raw.columns)
if missing:
    raise ValueError(f"Missing required CSV column(s): {sorted(missing)}")
if "user_agent" not in raw:
    raw["user_agent"] = ""
if raw["event_id"].duplicated().any():
    raise ValueError("Every event_id must be unique, so alerts can be joined reliably")

print(f"Loaded {len(raw)} login events from {raw['user_id'].nunique()} user IDs")
display(raw.head(8))

## 4. Parse timestamps and validate public IP addresses

Use Python's `ipaddress` module. `is_global` excludes loopback, private, reserved, and other non-public addresses. **Never geolocate an arbitrary or malformed client-supplied forwarded-IP header without a trusted reverse-proxy configuration.**

Invalid or non-public addresses remain in the exported report as *unassessable*, not safe.

In [ ]:
def inspect_ip(raw_ip):
    try:
        parsed = ipaddress.ip_address(str(raw_ip).strip())
    except ValueError:
        return None, "invalid_ip"
    normalized = str(parsed)
    return normalized, ("public" if parsed.is_global else "non_public")


login_events = raw.copy()
login_events.insert(0, "_row_id", range(len(login_events)))
login_events["user_id"] = login_events["user_id"].str.strip()
login_events["outcome"] = login_events["outcome"].str.strip().str.lower()
login_events["timestamp"] = pd.to_datetime(login_events["timestamp_utc"], errors="coerce", utc=True)
login_events[["ip_normalized", "ip_state"]] = pd.DataFrame(
    login_events["ip_address"].map(inspect_ip).tolist(),
    index=login_events.index,
)

invalid_outcomes = ~login_events["outcome"].isin({"success", "failure"})
if invalid_outcomes.any():
    raise ValueError(f"Unsupported 'outcome' values: {login_events.loc[invalid_outcomes, 'outcome'].unique().tolist()}")
if (login_events["user_id"] == "").any():
    raise ValueError("user_id must be nonempty")

print("IP validation:")
display(login_events.groupby("ip_state", dropna=False).size().rename("events").to_frame())
print("Unparseable timestamps:", int(login_events["timestamp"].isna().sum()))
display(login_events[["event_id", "user_id", "timestamp_utc", "ip_address", "ip_state", "outcome"]].head(10))

## 5. Offline **synthetic** geolocation fixtures

Every country, coordinate, and city in this dictionary is **fictional for teaching purposes**; there is no guarantee that it matches the actual public DNS address. The fixtures intentionally create cases of rapid long-distance movement and ordinary repeat sign-ins. **Do not use this dictionary as an IP geolocation dataset.**

In [ ]:
# Synthetic values, deliberately NOT the actual geolocation of these IPs.
DEMO_FAKE_LOCATIONS = {
    "8.8.8.8":                ("US", "United States", "California", "Los Angeles", 34.0522, -118.2437),
    "1.1.1.1":                ("GB", "United Kingdom", "England", "London", 51.5074, -0.1278),
    "9.9.9.9":                ("US", "United States", "New York", "New York", 40.7128, -74.0060),
    "208.67.222.222":         ("SG", "Singapore", "Singapore", "Singapore", 1.3521, 103.8198),
    "4.2.2.2":                ("AU", "Australia", "New South Wales", "Sydney", -33.8688, 151.2093),
    "2606:4700:4700::1111": ("JP", "Japan", "Tokyo", "Tokyo", 35.6762, 139.6503),
    "2001:4860:4860::8888": ("US", "United States", "California", "Los Angeles", 34.0522, -118.2437),
}

GEOGRAPHY_COLUMNS = ["country_code", "country_name", "region_name", "city_name", "latitude", "longitude"]


def demo_lookup(ip):
    values = DEMO_FAKE_LOCATIONS.get(ip)
    if values is None:
        return {"lookup_status": "missing_demo_fixture", "lookup_error": "No synthetic fixture for IP", "location_source": "synthetic_demo"}
    return {
        **dict(zip(GEOGRAPHY_COLUMNS, values)),
        "lookup_status": "ok",
        "lookup_error": "",
        "location_source": "synthetic_demo",
    }

print("Synthetic fixture coverage:", len(DEMO_FAKE_LOCATIONS), "distinct IP addresses")

## 6. Live IP2Location.io API lookup helper

[IP2Location.io REST API](https://www.ip2location.io/ip2location-documentation): `GET https://api.ip2location.io/?ip=...&format=json`, optional `Authorization: Bearer <API_KEY>`.

Supports IPv4/IPv6, JSON API error payloads, and network/HTTP error reporting. API failures remain *unknown*, never treated as a clean location. Avoid printing tokens or full URLs in shared logs. The keyless endpoint allows up to 1,000 queries/day (reset 00:00 UTC); a Free-plan key offers a different monthly quota. Quotas and available fields vary by plan.

In [ ]:
def api_lookup(ip, session):
    try:
        response = session.get(
            API_URL,
            params={"ip": ip, "format": "json"},
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
    except requests.RequestException as exc:
        return {"lookup_status": "network_error", "lookup_error": type(exc).__name__, "location_source": "ip2location.io_live"}

    if response.status_code == 429:
        return {"lookup_status": "rate_limited", "lookup_error": "HTTP 429", "location_source": "ip2location.io_live"}
    try:
        payload = response.json()
    except ValueError:
        return {"lookup_status": "invalid_json", "lookup_error": f"HTTP {response.status_code}", "location_source": "ip2location.io_live"}
    if not isinstance(payload, dict):
        return {"lookup_status": "unexpected_response", "lookup_error": "Expected a JSON object", "location_source": "ip2location.io_live"}
    if "error" in payload:
        detail = payload["error"] if isinstance(payload["error"], dict) else {}
        error_code = str(detail.get("error_code", "unknown"))
        return {
            "lookup_status": "api_error",
            "lookup_error": f"API error code {error_code}: {detail.get('error_message', 'no message')}",
            "location_source": "ip2location.io_live",
        }
    if not response.ok:
        return {"lookup_status": "http_error", "lookup_error": f"HTTP {response.status_code}", "location_source": "ip2location.io_live"}
    if not payload.get("country_code"):
        return {"lookup_status": "missing_geography", "lookup_error": "Country code unavailable", "location_source": "ip2location.io_live"}

    return {
        **{field: payload.get(field) for field in GEOGRAPHY_COLUMNS},
        "lookup_status": "ok",
        "lookup_error": "",
        "location_source": "ip2location.io_live",
    }

print("API lookup helper ready; no requests have been made yet.")

## 7. Enrich **unique** public IP addresses

In demo mode this performs no API requests; in live mode it requests each valid public IP once. Excessive input sizes are rejected **before any requests are sent**. If the API reports a quota issue (including HTTP 429 or API error 10000), processing stops and remaining IPs are marked unqueried; partial output can still be inspected.

In [ ]:
unique_public_ips = sorted(login_events.loc[login_events["ip_state"] == "public", "ip_normalized"].unique())
print(f"Public unique IPs: {len(unique_public_ips)}; total event rows: {len(login_events)}")

if MODE == "live" and len(unique_public_ips) > MAX_UNIQUE_IPS:
    raise ValueError(f"Live query count {len(unique_public_ips)} exceeds MAX_UNIQUE_IPS={MAX_UNIQUE_IPS}; reduce the input or adjust deliberately.")

results_by_ip = {}
if MODE == "demo":
    results_by_ip = {ip: demo_lookup(ip) for ip in unique_public_ips}
else:
    session = requests.Session()
    if API_KEY:
        session.headers.update({"Authorization": f"Bearer {API_KEY}"})
    session.headers.update({"Accept": "application/json"})
    quota_exhausted = False
    for ip in unique_public_ips:
        if quota_exhausted:
            results_by_ip[ip] = {
                "lookup_status": "not_queried_quota",
                "lookup_error": "Stopped following API quota or rate-limit response",
                "location_source": "ip2location.io_live",
            }
            continue
        result = api_lookup(ip, session)
        results_by_ip[ip] = result
        if result["lookup_status"] == "rate_limited" or (
            result["lookup_status"] == "api_error" and "API error code 10000:" in result.get("lookup_error", "")
        ):
            quota_exhausted = True
        if REQUEST_DELAY_SECONDS > 0 and not quota_exhausted:
            time.sleep(REQUEST_DELAY_SECONDS)
    session.close()

lookup_table = pd.DataFrame([
    {"ip_normalized": ip, **result}
    for ip, result in results_by_ip.items()
])
if lookup_table.empty:
    lookup_table = pd.DataFrame(columns=["ip_normalized", "lookup_status", "lookup_error", "location_source", *GEOGRAPHY_COLUMNS])

enriched = login_events.merge(lookup_table, on="ip_normalized", how="left", validate="many_to_one")
enriched["lookup_status"] = enriched["lookup_status"].fillna(
    enriched["ip_state"].map({"invalid_ip": "skipped_invalid_ip", "non_public": "skipped_non_public"})
)
enriched["lookup_error"] = enriched["lookup_error"].fillna("")
enriched["location_source"] = enriched["location_source"].fillna("none")
for coord in ("latitude", "longitude"):
    if coord not in enriched.columns:
        enriched[coord] = float("nan")
    enriched[coord] = pd.to_numeric(enriched[coord], errors="coerce")
for column in ("country_code", "country_name", "region_name", "city_name"):
    if column not in enriched:
        enriched[column] = None

print("Lookup outcomes:")
display(enriched.groupby("lookup_status", dropna=False).size().rename("events").to_frame())
display(enriched[["event_id", "ip_address", "country_code", "city_name", "lookup_status", "location_source"]].head(12))

## 8. Compute great-circle distance and travel velocity

For each user, compare the newest *successful, timestamped, geolocated* login to their **previous successful geolocated login**. Failed login attempts do not replace that baseline. Missing/invalid geographic information remains unassessable, while a first successful login is a baseline.

A **high** review flag requires (1) at least 500 km of displacement, (2) more than 900 km/h implied speed, and (3) two positive-time logins no more than 24 hours apart. A **medium** flag represents a change of country without high-speed travel. Thresholds are illustrative and should be tuned to your traffic patterns, not treated as universal security policy.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle separation in kilometres, not actual travel-route distance."""
    radius_km = 6371.0088
    a1, a2 = math.radians(lat1), math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    h = math.sin(dlat / 2) ** 2 + math.cos(a1) * math.cos(a2) * math.sin(dlon / 2) ** 2
    return 2 * radius_km * math.asin(min(1.0, math.sqrt(max(0.0, h))))


def has_coordinates(row):
    return pd.notna(row["latitude"]) and pd.notna(row["longitude"]) and (
        -90 <= row["latitude"] <= 90 and -180 <= row["longitude"] <= 180
    )


scored = []
ordered = enriched.sort_values(["user_id", "timestamp", "_row_id"], na_position="last")
for user_id, group in ordered.groupby("user_id", sort=False):
    previous = None
    for _, event in group.iterrows():
        record = {
            "_row_id": event["_row_id"],
            "previous_event_id": None,
            "hours_since_previous": None,
            "distance_km": None,
            "implied_speed_kmh": None,
            "country_changed": False,
            "impossible_travel": False,
            "review_level": "none",
            "review_reason": "",
        }
        if event["outcome"] != "success":
            record.update(review_level="not_scored", review_reason="Failed login: not a location baseline")
        elif pd.isna(event["timestamp"]):
            record.update(review_level="unassessable", review_reason="Missing or invalid UTC timestamp")
        elif event["lookup_status"] != "ok" or not has_coordinates(event):
            record.update(review_level="unassessable", review_reason="IP geolocation unavailable or non-public")
        else:
            if previous is None:
                record.update(review_level="baseline", review_reason="First comparable successful login")
            else:
                record["previous_event_id"] = previous["event_id"]
                delta_hours = (event["timestamp"] - previous["timestamp"]).total_seconds() / 3600
                record["hours_since_previous"] = round(delta_hours, 4)
                if delta_hours <= 0:
                    record.update(review_level="unassessable", review_reason="Non-positive time gap")
                elif delta_hours > MAX_COMPARISON_HOURS:
                    record.update(review_level="none", review_reason="Beyond configured comparison window")
                else:
                    distance = haversine_km(
                        previous["latitude"], previous["longitude"], event["latitude"], event["longitude"]
                    )
                    speed = distance / delta_hours
                    record["distance_km"] = round(distance, 1)
                    record["implied_speed_kmh"] = round(speed, 1)
                    before_code = previous["country_code"]
                    after_code = event["country_code"]
                    changed = pd.notna(before_code) and pd.notna(after_code) and str(before_code) != str(after_code)
                    record["country_changed"] = bool(changed)
                    record["impossible_travel"] = bool(
                        distance >= MIN_DISTANCE_KM and speed > MAX_REASONABLE_SPEED_KMH
                    )
                    if record["impossible_travel"]:
                        record.update(review_level="high", review_reason="Improbable travel speed (manual review)")
                    elif changed:
                        record.update(review_level="medium", review_reason="Country change (manual review)")
                    else:
                        record.update(review_level="none", review_reason="No configured travel flag")

            # Only eligible successes update the comparison anchor.
            previous = event
        scored.append(record)

flags = pd.DataFrame(scored)
review = enriched.merge(flags, on="_row_id", how="left", validate="one_to_one")
review = review.sort_values("_row_id").reset_index(drop=True)
print("Review-level counts (includes failed logins and unassessable events):")
display(review.groupby("review_level").size().rename("events").to_frame())

## 9. Investigate high- and medium-priority events

Alerts show the relevant previous successful event and the great-circle distance/elapsed-time calculation. The classification is an explanation for *human review*, not an authorization decision.

In [ ]:
alerts = review.loc[review["review_level"].isin(["high", "medium"])].copy()
alert_cols = [
    "event_id", "user_id", "timestamp_utc", "ip_address", "country_code", "city_name",
    "previous_event_id", "hours_since_previous", "distance_km", "implied_speed_kmh",
    "country_changed", "impossible_travel", "review_level", "review_reason", "location_source"
]
print(f"Review alerts: {len(alerts)}")
display(alerts[alert_cols].sort_values(["review_level", "user_id", "timestamp_utc"]))
if MODE == "demo":
    assert (alerts["review_level"] == "high").sum() >= 2, "Expected two or more synthetic high-speed examples"
    assert (review.loc[review["ip_state"] != "public", "review_level"] == "unassessable").all()
    print("Offline demonstration checks passed.")

## 10. Account-level summary

Counts are event counts, **not** verified fraud cases. High and medium alerts should be reviewed alongside device fingerprint, MFA challenge outcome, session history, known VPN use, and user confirmation.

In [ ]:
user_summary = (review.groupby("user_id", dropna=False)
    .agg(
        login_events=("event_id", "count"),
        successful_logins=("outcome", lambda s: int((s == "success").sum())),
        public_unique_ips=("ip_normalized", lambda s: int(s.dropna().nunique())),
        high_alerts=("review_level", lambda s: int((s == "high").sum())),
        medium_alerts=("review_level", lambda s: int((s == "medium").sum())),
        unassessable_events=("review_level", lambda s: int((s == "unassessable").sum())),
    )
    .reset_index()
    .sort_values(["high_alerts", "medium_alerts", "user_id"], ascending=[False, False, True])
)
display(user_summary)

## 11. Plot the review distribution

Charts use the chosen mode's enrichment results. In demo mode all locations and resulting travel alerts are **synthetic illustrations**.

In [ ]:
categories = ["high", "medium", "none", "baseline", "unassessable", "not_scored"]
counts = review["review_level"].value_counts().reindex(categories, fill_value=0)
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(counts.index, counts.values)
ax.set_title("Login review categories" + (" (synthetic demonstration)" if MODE == "demo" else " (live API)"))
ax.set_xlabel("Review category")
ax.set_ylabel("Events")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
plt.show()

## 12. Export enriched events and alert reports

Original CSV fields are retained; additional fields document the geolocation source, comparison baseline, distance, speed and flag reason. Consider access controls, retention limits, and IP pseudonymization before storing real account data.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
export_events = OUTPUT_DIR / "login_events_enriched.csv"
export_alerts = OUTPUT_DIR / "login_travel_alerts.csv"
export_users = OUTPUT_DIR / "login_user_summary.csv"

review.drop(columns=["_row_id", "timestamp"], errors="ignore").to_csv(export_events, index=False)
alerts[alert_cols].to_csv(export_alerts, index=False)
user_summary.to_csv(export_users, index=False)

for file in (export_events, export_alerts, export_users):
    print(f"Saved {file} ({file.stat().st_size:,} bytes)")

## 13. Operational caveats and next steps

- **Geolocation uncertainty:** IP-derived latitude/longitude often represent city centroids; if a city is unknown, [IP2Location.io may return capital-city coordinates](https://www.ip2location.io/ip2location-documentation). Distance and speed are *rough approximations*.
- **Proxy / VPN / anycast:** A geo jump is not reliable evidence the user traveled; upstream infrastructure, carrier egress and shared addresses can explain it. Account takeover investigations need additional signals.
- **Traffic provenance:** Capture the actual client IP through a correctly configured trusted proxy; do not trust arbitrary `X-Forwarded-For` values from clients.
- **Time normalization:** Convert all event timestamps to UTC before analysis. Sort events per user. Missing timestamps or coordinates must remain unassessable.
- **Fraud prevention:** Review with MFA/device/session signals and allow user verification; avoid hard blocks or automated accusations from this notebook alone.
- **API limits:** Live mode performs one API request per unique public IP, with a configurable upper bound. Use secure key storage and a cache/retention policy when scaling.

**Documentation:** [IP2Location.io API](https://www.ip2location.io/ip2location-documentation) · [IP2Location.io pricing/limits](https://www.ip2location.io/pricing)

**Next notebook idea (06/10):** Localize website content or offers based on the visitor's IP location, with a sensible fallback when geolocation is unknown.